# Agent 3: Response Generator — QLoRA Fine-Tuning

Fine-tune LLaMA-3.2-3B-Instruct to generate appointment confirmations & pre-visit instructions.

**Updated**: Emergency urgency is now a first-class level (Routine / Urgent / Emergency).

**Runtime**: T4 GPU on Colab, ~2-3 hours for full training.

## Cell 1: Setup & Dependencies

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl huggingface_hub

## Cell 2: Imports & HuggingFace Login

In [ ]:
import json
import random
import re
import os
from collections import Counter, defaultdict

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from google.colab import userdata, files

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

login(token=userdata.get('HF_TOKEN'))

## Cell 3: Configuration

In [ ]:
# ── Model ──
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

# ── Data Processing ──
SAMPLE_RATIO = 0.20
MIN_RESPONSE_LEN = 80
MAX_RESPONSE_LEN = 800
MAX_SYMPTOM_LEN = 200
TRAIN_RATIO = 0.85

# ── QLoRA ──
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# ── Training ──
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 512
WARMUP_RATIO = 0.03

# ── Paths (Google Drive) ──
OUTPUT_DIR = "/content/drive/MyDrive/response_generator"
DATA_DIR = "/content/drive/MyDrive/response_generator"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Cell 4: Upload doctor_schedules.json

Upload from your local `data/processed/doctor_schedules.json`.

In [ ]:
schedule_path = os.path.join(DATA_DIR, "doctor_schedules.json")
if not os.path.exists(schedule_path):
    print("Please upload doctor_schedules.json")
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(schedule_path, 'wb') as f:
            f.write(content)
    print(f"Saved to {schedule_path}")
else:
    print(f"Already exists: {schedule_path}")

with open(schedule_path) as f:
    schedules = json.load(f)

n_doctors = len(set(r["doctor"] for r in schedules))
n_depts = len(set(r["department"] for r in schedules))
print(f"Loaded {n_doctors} doctors across {n_depts} departments, {len(schedules)} total slots")

## Cell 5: Data Processing — Build Training Data with 3-Level Urgency

In [ ]:
INSTRUCTION_TEMPLATE = (
    "You are a compassionate medical assistant. "
    "A patient has been assigned an appointment. "
    "Write a warm, clear appointment confirmation and practical pre-visit instructions. "
    "Keep the tone professional but reassuring. "
    "Format your response as:\n"
    "Confirmation: <one sentence confirming the appointment>\n"
    "Instructions: <2-4 specific pre-visit instructions>"
)

EMERGENCY_DEPARTMENTS = {"Cardiology", "Neurology", "Pulmonology", "Infectious Disease"}


def assign_urgency(department):
    if department in EMERGENCY_DEPARTMENTS:
        return random.choices(["Routine", "Urgent", "Emergency"], weights=[30, 40, 30], k=1)[0]
    else:
        return random.choices(["Routine", "Urgent", "Emergency"], weights=[45, 40, 15], k=1)[0]


def build_synthetic_context(department, doctor, time_slot, urgency, symptoms):
    return (
        f"Patient symptoms: {symptoms}\n"
        f"Assigned department: {department}\n"
        f"Doctor: {doctor}\n"
        f"Appointment: {time_slot}\n"
        f"Urgency: {urgency}"
    )


def dialogue_to_record(patient_q, doctor_a, schedules):
    doctor_a = str(doctor_a).strip()
    if len(doctor_a) < MIN_RESPONSE_LEN or len(doctor_a) > MAX_RESPONSE_LEN:
        return None

    entry = random.choice(schedules)
    department = entry["department"]
    doctor = entry["doctor"]
    time_slot = f"{entry['day']} at {entry['time_slot']}"
    urgency = assign_urgency(department)
    symptoms = str(patient_q).strip()[:MAX_SYMPTOM_LEN]

    return {
        "instruction": INSTRUCTION_TEMPLATE,
        "input": build_synthetic_context(department, doctor, time_slot, urgency, symptoms),
        "output": doctor_a,
        "department": department,
        "urgency": urgency,
    }


# Load & process
random.seed(42)

print("Loading AI Medical Chatbot from HuggingFace...")
ds = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
print(f"Loaded {len(ds)} dialogues")

indices = random.sample(range(len(ds)), int(len(ds) * SAMPLE_RATIO))
sampled = ds.select(indices)
print(f"Sampled {len(sampled)} rows ({SAMPLE_RATIO:.0%})")

records = []
for row in sampled:
    rec = dialogue_to_record(
        patient_q=row.get("Patient", row.get("input", "")),
        doctor_a=row.get("Doctor", row.get("output", "")),
        schedules=schedules,
    )
    if rec:
        records.append(rec)

print(f"Kept {len(records)} records after quality filtering")

# Distribution check
urg_counts = Counter(r["urgency"] for r in records)
dept_counts = Counter(r["department"] for r in records)
print(f"\nUrgency distribution:")
for u, c in sorted(urg_counts.items()):
    print(f"  {u:<12} {c:>6} ({c/len(records):.1%})")
print(f"\nDepartment distribution:")
for d, c in sorted(dept_counts.items()):
    print(f"  {d:<22} {c:>6} ({c/len(records):.1%})")

## Cell 6: Train/Test Split & Save JSONL

In [ ]:
random.seed(42)
random.shuffle(records)
split = int(len(records) * TRAIN_RATIO)
train_records, test_records = records[:split], records[split:]

train_path = os.path.join(DATA_DIR, "response_generator_train.jsonl")
test_path = os.path.join(DATA_DIR, "response_generator_test.jsonl")

for path, data in [(train_path, train_records), (test_path, test_records)]:
    with open(path, "w") as f:
        for r in data:
            f.write(json.dumps(r) + "\n")

print(f"Train: {len(train_records)} | Test: {len(test_records)}")
print(f"Saved to {DATA_DIR}/")

## Cell 7: Format Data for SFTTrainer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def format_chat(record):
    messages = [
        {"role": "system", "content": record["instruction"]},
        {"role": "user", "content": record["input"]},
        {"role": "assistant", "content": record["output"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)


train_dataset = Dataset.from_list(train_records)
train_dataset = train_dataset.map(
    lambda x: {"text": format_chat(x)},
    remove_columns=train_dataset.column_names,
)

test_dataset = Dataset.from_list(test_records)
test_dataset = test_dataset.map(
    lambda x: {"text": format_chat(x)},
    remove_columns=test_dataset.column_names,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print(f"\nExample:\n{train_dataset[0]['text'][:500]}")

## Cell 8: Load Model with QLoRA

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 9: Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    args=training_args,
    max_seq_length=MAX_SEQ_LEN,
)

print("Starting training...")
trainer.train()

## Cell 10: Save Adapter

In [ ]:
final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Adapter saved to {final_adapter_path}")

## Cell 11: Inference Test

In [ ]:
def parse_response_output(text):
    conf_match = re.search(r"Confirmation:\s*(.+?)(?=Instructions:|$)", text, re.DOTALL)
    inst_match = re.search(r"Instructions:\s*(.+)", text, re.DOTALL)
    return {
        "confirmation": conf_match.group(1).strip() if conf_match else text.strip(),
        "instructions": inst_match.group(1).strip() if inst_match else "",
    }


def generate_response(model, tokenizer, patient_text, department, doctor, time_slot, urgency):
    user_content = (
        f"Patient symptoms: {patient_text}\n"
        f"Assigned department: {department}\n"
        f"Doctor: {doctor}\n"
        f"Appointment: {time_slot}\n"
        f"Urgency: {urgency}"
    )
    messages = [
        {"role": "system", "content": INSTRUCTION_TEMPLATE},
        {"role": "user", "content": user_content},
    ]
    tokenized = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    input_len = tokenized["input_ids"].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(
            **tokenized,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return parse_response_output(generated)


# Test with all 3 urgency levels
test_cases = [
    {
        "patient_text": "I have been having chest pain and shortness of breath for 3 days.",
        "department": "Cardiology",
        "doctor": "Dr. Chen Wei",
        "time_slot": "Monday at 08:00",
        "urgency": "Emergency",
    },
    {
        "patient_text": "I have a persistent headache and occasional dizziness for a week.",
        "department": "Neurology",
        "doctor": "Dr. Sarah Johnson",
        "time_slot": "Wednesday at 10:00",
        "urgency": "Urgent",
    },
    {
        "patient_text": "My skin has been itchy with small red patches.",
        "department": "Dermatology",
        "doctor": "Dr. Emily Brown",
        "time_slot": "Friday at 14:00",
        "urgency": "Routine",
    },
]

for tc in test_cases:
    print(f"\n{'='*60}")
    print(f"Urgency: {tc['urgency']} | Dept: {tc['department']}")
    print(f"{'='*60}")
    result = generate_response(model, tokenizer, **tc)
    print(f"Confirmation: {result['confirmation']}")
    print(f"Instructions: {result['instructions']}")

## Cell 12: Evaluation on Test Set

In [ ]:
def evaluate_model(model, tokenizer, test_path, limit=None):
    with open(test_path) as f:
        data = [json.loads(line) for line in f]
    if limit:
        data = data[:limit]

    total = len(data)
    format_ok = 0
    has_confirmation = 0
    has_instructions = 0
    total_length = 0

    dept_counts = defaultdict(lambda: {"total": 0, "format_ok": 0})
    urg_counts = defaultdict(lambda: {"total": 0, "format_ok": 0})

    for i, sample in enumerate(data):
        input_text = sample["input"]
        fields = {}
        for line in input_text.strip().split("\n"):
            if line.startswith("Patient symptoms:"):
                fields["patient_text"] = line.replace("Patient symptoms:", "").strip()
            elif line.startswith("Assigned department:"):
                fields["department"] = line.replace("Assigned department:", "").strip()
            elif line.startswith("Doctor:"):
                fields["doctor"] = line.replace("Doctor:", "").strip()
            elif line.startswith("Appointment:"):
                fields["time_slot"] = line.replace("Appointment:", "").strip()
            elif line.startswith("Urgency:"):
                fields["urgency"] = line.replace("Urgency:", "").strip()

        result = generate_response(
            model, tokenizer,
            patient_text=fields.get("patient_text", ""),
            department=fields.get("department", "General Medicine"),
            doctor=fields.get("doctor", "Dr. Unknown"),
            time_slot=fields.get("time_slot", "Monday at 08:00"),
            urgency=fields.get("urgency", "Routine"),
        )

        conf = result["confirmation"].strip()
        inst = result["instructions"].strip()
        both = bool(conf) and bool(inst)

        if both:
            format_ok += 1
        if conf:
            has_confirmation += 1
        if inst:
            has_instructions += 1
        total_length += len(f"{conf} {inst}")

        dept = fields.get("department", "Unknown")
        urg = fields.get("urgency", "Unknown")
        dept_counts[dept]["total"] += 1
        urg_counts[urg]["total"] += 1
        if both:
            dept_counts[dept]["format_ok"] += 1
            urg_counts[urg]["format_ok"] += 1

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{total}] format compliance: {format_ok/(i+1):.1%}")

    print(f"\n{'='*60}")
    print("RESPONSE GENERATOR EVALUATION REPORT")
    print(f"{'='*60}")
    print(f"Total samples:         {total}")
    print(f"Format compliance:     {format_ok/total:.1%}")
    print(f"Confirmation present:  {has_confirmation/total:.1%}")
    print(f"Instructions present:  {has_instructions/total:.1%}")
    print(f"Avg response length:   {total_length/total:.0f} chars")

    print(f"\n--- Per-Urgency ---")
    print(f"{'Urgency':<15} {'Compliance':>10} {'Samples':>8}")
    for urg, c in sorted(urg_counts.items()):
        comp = c['format_ok'] / c['total'] if c['total'] > 0 else 0
        print(f"{urg:<15} {comp:>10.1%} {c['total']:>8}")

    print(f"\n--- Per-Department ---")
    print(f"{'Department':<22} {'Compliance':>10} {'Samples':>8}")
    for dept, c in sorted(dept_counts.items()):
        comp = c['format_ok'] / c['total'] if c['total'] > 0 else 0
        print(f"{dept:<22} {comp:>10.1%} {c['total']:>8}")

    return {
        "total": total,
        "format_compliance": format_ok / total,
        "confirmation_rate": has_confirmation / total,
        "instructions_rate": has_instructions / total,
    }


results = evaluate_model(model, tokenizer, test_path, limit=200)

with open(os.path.join(DATA_DIR, "eval_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {DATA_DIR}/eval_results.json")

## Cell 13: Download Adapter

In [ ]:
final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
!zip -r /content/response_generator_output.zip {final_adapter_path}

files.download("/content/response_generator_output.zip")
print("Download complete! Unzip and place:")
print("  final_adapter/ -> data/processed/response_generator_adapter/final_adapter/")